# Which gene-level findings are neighbour-structure markers? A cell-type specificity filter (notebook 109)

**The question.** Notebook 104's core rebuild showed gradients and AKI calls survive when units are rebuilt from bins
≥ 6 µm inside the polygon edge. Notebook 108 found distal-disease genes graded inside PT units that also survive it
(SLC12A1, SLC12A3, AQP2, KCNJ1, SCNN1A). A single-nucleus atlas filter is the missing check: a gene whose expression
sits mainly outside the PT is a neighbour marker.

**The frozen protocol** is `docs/results/pt-neighbour-marker-filter.md` (commit c55224c).
- Neighbour marker: max non-PT CP10k ≥ 1 and ≥ 4 × PT CP10k (Kirita control nuclei for mouse; KPMP healthy-reference
  nuclei for human).
- R1: share flagged per list against an expression-matched null (clean ≤ 5%). R2: agreement with 104's edge
  enrichment (AUC ≥ 0.6). R3: each method's exposure. R4: the flag list.

**Sections.** 1 setup; 2 atlases (cached); 3 specificity; 4 lists and R1; 5 R2; 6 R3; 7 R4 and verdicts; 8 figures;
9 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
KPMP = data_root / 'external' / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
C105 = results_root / 'pt_external_spatial_iri' / 'stage_cache'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T101 = results_root / 'pt_transporter_handoffs' / 'tables'
T108 = results_root / 'pt_disease_genetics' / 'tables'
T93 = results_root / 'pt_pathway_networks' / 'tables'
T76 = results_root / 'pt_package_aki' / 'tables'
T71 = results_root / 'pt_pathways_package_pseudospace' / 'tables'
T104 = results_root / 'pt_contamination_audit' / 'tables'
T62 = results_root / 'pt_non_one_to_one_orthologs' / 'tables'
output = results_root / 'pt_neighbour_marker_filter'
tables, figures, cache_root = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '109.neighbour_filter.1'
SEED = 109
MIN_CP10K, FOLD, CLEAN_MAX, AUC_MIN, N_NULL, GRADED = 1.0, 4.0, .05, .6, 2000, .5
REFERENCE = ['Healthy_reference_tissue', 'Healthy_stone_donor', 'Tumor_nephrectomy']
HUMAN_TYPES = ['PT', 'TAL', 'DTL', 'ATL', 'DCT', 'CNT', 'PC', 'IC', 'POD', 'PEC', 'EC', 'FIB', 'IMM', 'VSM/P']
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
inputs = [KPMP, T90 / 'd_end_contrasts.csv.gz', T91 / 'i_genes.csv.gz', T101 / 'pairs.csv', T108 / 'h4_named.csv',
          T93 / 'calls_aki_vs_control.csv', T93 / 'calls_human_vs_mouse.csv', T104 / 'edge_enrichment.csv.gz',
          T62 / 'gene_inventory_mouse.csv', T62 / 'gene_inventory_human.csv', T76 / 'positive_control_markers.csv']
for path in inputs:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')
if not list(C105.glob('kirita_reference__*.npz')):
    raise FileNotFoundError('Run notebook 105 first: its cached Kirita reference is read here')

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from sklearn.metrics import roc_auc_score
from IPython.display import display

from pseudospace.external_replication import h5_group_sums, matched_null
from pseudospace.stage_cache import cached_payload

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPECIES_COLOR = {'mouse': '#0072B2', 'human': '#009E73'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 100)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []

## 2 · Atlases

In [ ]:
f105 = sorted(C105.glob('kirita_reference__*.npz'))[-1]
with np.load(f105, allow_pickle=False) as p:
    ksum = pd.DataFrame(p['sums'], index=p['genes'].astype(str), columns=p['columns'].astype(str))
ksum = ksum.groupby(level=0).sum()
ctrl = ksum[[c for c in ksum.columns if c.startswith('Control|')]]
ctrl.columns = ctrl.columns.str.split('|').str[1]
MOUSE_ATLAS = ctrl.drop(columns=['NewPT'], errors='ignore')
MOUSE_ATLAS = MOUSE_ATLAS.assign(PT=MOUSE_ATLAS[['PTS1', 'PTS2', 'PTS3']].sum(axis=1)).drop(columns=['PTS1', 'PTS2', 'PTS3'])
guard_log.append(f'G1: mouse atlas = Kirita control nuclei, {MOUSE_ATLAS.shape[1]} broad types incl. PT, {len(MOUSE_ATLAS):,} genes')

kp = ad.read_h5ad(KPMP, backed='r')
OBS = kp.obs[['disease_category', 'subclass.l1']].astype(str).copy()
HUMAN_GENES = pd.Index(kp.var['feature_name'].astype(str))
del kp
keep = OBS.disease_category.isin(REFERENCE) & OBS['subclass.l1'].isin(HUMAN_TYPES)


def human_atlas():
    rows = np.flatnonzero(keep.to_numpy())
    sums = h5_group_sums(KPMP, rows, OBS['subclass.l1'].to_numpy()[rows])
    return {'sums': sums.to_numpy(), 'types': np.array(sums.columns.astype(str)).astype('U')}


t0 = time.perf_counter()
HA = cached_payload('kpmp_reference_types', human_atlas, root=cache_root,
                    params={'logic': NOTEBOOK_LOGIC_VERSION, 'reference': REFERENCE, 'types': HUMAN_TYPES},
                    inputs={'kpmp': str(KPMP)}, code=h5_group_sums)
HUMAN_ATLAS = pd.DataFrame(HA['sums'], index=HUMAN_GENES, columns=HA['types'].astype(str)).groupby(level=0).sum()
print(f'Human atlas {time.perf_counter() - t0:.0f} s; nuclei per type:', OBS[keep]['subclass.l1'].value_counts().to_dict())
guard_log.append(f'G2: human atlas = KPMP {int(keep.sum()):,} reference nuclei over {len(HUMAN_TYPES)} subclasses')

## 3 · Specificity

In [ ]:
def specificity(atlas):
    cp = atlas / atlas.sum() * 1e4
    other = cp.drop(columns='PT')
    top = other.idxmax(axis=1)
    mx = other.max(axis=1)
    out = pd.DataFrame({'PT CP10k': cp.PT, 'top non-PT type': top, 'top non-PT CP10k': mx,
                        'ratio': mx / cp.PT.replace(0, np.nan)})
    out['neighbour marker'] = (mx >= MIN_CP10K) & (mx >= FOLD * cp.PT)
    return out


SPEC = {'mouse': specificity(MOUSE_ATLAS), 'human': specificity(HUMAN_ATLAS)}
pd.concat({k: v for k, v in SPEC.items()}, names=['species', 'gene']).to_csv(tables / 'atlas_specificity.csv.gz')
for sp, t in SPEC.items():
    print(sp, f"{int(t['neighbour marker'].sum()):,} neighbour markers of {len(t):,} genes;",
          'checks:', t.reindex(['Slc12a1', 'Slc12a3', 'Umod', 'Lrp2', 'Slc34a1'] if sp == 'mouse' else
                               ['SLC12A1', 'SLC12A3', 'UMOD', 'LRP2', 'SLC34A1'])['neighbour marker'].to_dict())

## 4 · Lists and R1

In [ ]:
END = pd.read_csv(T90 / 'd_end_contrasts.csv.gz')
END['human'] = END.human.astype(str)
for sp, pair in (('mouse', MOUSE), ('human', HUMAN)):
    END[f'early {sp}'] = (END[[f'E {s}' for s in pair]] <= -GRADED).all(axis=1)
    END[f'late {sp}'] = (END[[f'E {s}' for s in pair]] >= GRADED).all(axis=1)
inv = {'mouse': pd.read_csv(T62 / 'gene_inventory_mouse.csv').drop_duplicates('gene').set_index('gene').log2_cp10k,
       'human': pd.read_csv(T62 / 'gene_inventory_human.csv').drop_duplicates('gene').set_index('gene').log2_cp10k}
G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
pairs = pd.read_csv(T101 / 'pairs.csv')
pm, ph = (set(zip(pairs[pairs.species.eq(s)].early, pairs[pairs.species.eq(s)].late)) for s in ('mouse', 'human'))
conserved_pairs = pm & ph
handoff_mouse = sorted({g for p in conserved_pairs for g in p})
h4 = pd.read_csv(T108 / 'h4_named.csv')
m2h = dict(zip(END.mouse, END.human))
h2m = dict(zip(END.human, END.mouse))
markers76 = pd.read_csv(T76 / 'positive_control_markers.csv', index_col=0).index.astype(str).tolist()
sets76 = json.loads((T76 / 'gene_sets.json').read_text())
u76 = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
cyc = sorted((set(sets76['MSigDB_Hallmark_2020::G2-M Checkpoint']) | set(sets76['MSigDB_Hallmark_2020::E2F Targets'])) & u76)
tr = pd.read_csv(T76 / 'programs_transporters.csv')
tpt = sorted(tr.mouse[tr.in_T_PT.astype(bool)].astype(str))

LISTS = []   # (name, species, genes in that species' symbols, universe in that species' symbols)
uni_m, uni_h = list(END.mouse), list(END.human)
for sp, uni, key in (('mouse', uni_m, 'mouse'), ('human', uni_h, 'human')):
    LISTS.append((f'graded early ({sp})', sp, list(END.loc[END[f'early {sp}'], key]), uni))
    LISTS.append((f'graded late ({sp})', sp, list(END.loc[END[f'late {sp}'], key]), uni))
    for cat in ('conserved', 'reversed', 'mouse only', 'human only'):
        LISTS.append((f'{cat} ({sp} atlas)', sp, list(END.loc[END.category.eq(cat), key]), uni))
    LISTS.append((f'conserved handoff genes ({sp} atlas)', sp, handoff_mouse if sp == 'mouse' else [m2h.get(g, g) for g in handoff_mouse], uni))
    LISTS.append((f'named positions H4 ({sp} atlas)', sp, sorted(set(h4.gene)) if sp == 'human' else sorted({h2m.get(g, g) for g in h4.gene}), uni))
uni_aki = list(G91.index.astype(str))
LISTS += [('AKI level calls (91)', 'mouse', list(G91.index[G91['level change'].astype(bool)]), uni_aki),
          ('AKI location calls (91)', 'mouse', list(G91.index[G91['location change'].astype(bool)]), uni_aki),
          ('program INJ', 'mouse', markers76, uni_aki), ('program TPT', 'mouse', tpt, uni_aki), ('program CYC', 'mouse', cyc, uni_aki)]


def screen(name, sp, genes, universe, seed):
    spec = SPEC[sp]
    uni = pd.Index([g for g in dict.fromkeys(universe) if g in spec.index])
    genes = [g for g in dict.fromkeys(genes) if g in uni]
    level = inv[sp].reindex(uni) if sp in inv else pd.Series(np.nan, index=uni)
    if sp == 'mouse' and level.isna().mean() > .5:
        level = pd.Series(np.log1p(SPEC['mouse'].loc[uni, 'PT CP10k']).to_numpy(), index=uni)
    decile = pd.qcut(level.rank(method='first'), 10, labels=False).fillna(-1).astype(int)
    hit = spec.loc[uni, 'neighbour marker'].to_numpy()
    flagged = [g for g in genes if spec.loc[g, 'neighbour marker']]
    null = matched_null(hit, decile.to_numpy(), decile.reindex(genes).to_numpy(), n=N_NULL, seed=seed) if genes else np.zeros(1)
    share = len(flagged) / max(len(genes), 1)
    return {'list': name, 'species atlas': sp, 'assessable': len(genes), 'flagged': len(flagged), 'share': share,
            'null mean': float(null.mean() / max(len(genes), 1)), 'null 95%': float(np.quantile(null, .95) / max(len(genes), 1)),
            'clean': bool(share <= CLEAN_MAX), 'flagged genes': ', '.join(flagged)}


R1 = pd.DataFrame([screen(n, sp, g, u, SEED + i) for i, (n, sp, g, u) in enumerate(LISTS)])
R1.to_csv(tables / 'lists.csv', index=False)
display(R1.drop(columns='flagged genes').round(3))

## 5 · R2: agreement with notebook 104's edge enrichment

In [ ]:
eps = pd.read_csv(T104 / 'edge_enrichment.csv.gz', index_col=0)
r2 = []
for sp, pair in (('mouse', MOUSE), ('human', HUMAN)):
    e = eps[[f'hvm {s}' for s in pair]].mean(axis=1)
    lab = SPEC[sp]['neighbour marker']
    if sp == 'human':
        e.index = [m2h.get(g, None) for g in e.index]
        e = e[e.index.notna()]
        e = e[~e.index.duplicated()]
    e = e.dropna()
    common = e.index.intersection(lab.index)
    auc = roc_auc_score(lab[common].astype(int), e[common])
    r2.append({'species': sp, 'genes': len(common), 'neighbour markers': int(lab[common].sum()), 'AUC of ε': auc,
               'median ε markers': float(e[common][lab[common]].median()), 'median ε others': float(e[common][~lab[common]].median()),
               'agree': bool(auc >= AUC_MIN)})
R2 = pd.DataFrame(r2)
R2.to_csv(tables / 'r2_agreement.csv', index=False)
display(R2.round(3))

### 5.1 · Post hoc: neighbour markers that are also edge-enriched

Added after R1 found ~10–28% of most lists flagged, near the expression-matched null: the atlas rule alone flags any
gene expressed ≥ 4× more in some other cell type, which does not mean its PT gradient is neighbour RNA. A gene is
**suspect** when it is a neighbour marker **and** edge-enriched in notebook 104 (ε ≥ 1 in both specimens of the
species, human vs mouse run; AKI lists use the AKI run's four mice).

In [ ]:
def edge_flag(sp):
    if sp == 'human':
        e = eps[[f'hvm {s}' for s in HUMAN]]
        ok = (e >= 1).all(axis=1)
        ok.index = [m2h.get(g, None) for g in ok.index]
        ok = ok[ok.index.notna()]
        return ok[~ok.index.duplicated()]
    return (eps[[f'hvm {s}' for s in MOUSE]] >= 1).all(axis=1)


EDGE = {'mouse': edge_flag('mouse'), 'human': edge_flag('human'),
        'aki': (eps[[c for c in eps.columns if c.startswith('aki ')]] >= 1).all(axis=1)}
susp = []
for (name, sp, genes, universe), (_, r) in zip(LISTS, R1.iterrows()):
    edge = EDGE['aki'] if name.startswith(('AKI', 'program')) else EDGE[sp]
    flagged = [g for g in str(r['flagged genes']).split(', ') if g and g != 'nan']
    s_genes = [g for g in flagged if bool(edge.get(g, False))]
    susp.append({'list': name, 'assessable': r['assessable'], 'suspect': len(s_genes), 'suspect share': len(s_genes) / max(r['assessable'], 1),
                 'suspect genes': ', '.join(s_genes)})
SUSPECT = pd.DataFrame(susp)
SUSPECT.to_csv(tables / 'posthoc_suspect.csv', index=False)
display(SUSPECT.round(3))

## 6 · R3: each method's exposure to neighbour markers

In [ ]:
r3 = []
for contrast, universe_file in (('aki_vs_control', 'universe_aki_vs_control.csv'), ('human_vs_mouse', 'universe_human_vs_mouse.csv')):
    uni = pd.Index(pd.read_csv(T71 / universe_file).gene.astype(str))
    uni = uni[uni.isin(SPEC['mouse'].index)]
    if contrast == 'aki_vs_control':
        hit = SPEC['mouse'].loc[uni, 'neighbour marker']
    else:
        hum = pd.Series([m2h.get(g) for g in uni], index=uni)
        hit = SPEC['mouse'].loc[uni, 'neighbour marker'] | hum.map(lambda h: bool(SPEC['human']['neighbour marker'].get(h, False)) if h else False)
    level = pd.Series(np.log1p(SPEC['mouse'].loc[uni, 'PT CP10k']).to_numpy(), index=uni)
    decile = pd.qcut(level.rank(method='first'), 10, labels=False)
    calls = pd.read_csv(T93 / f'calls_{contrast}.csv')
    for method, g in calls.groupby('method'):
        drivers = pd.Index(sorted(set(','.join(g['driver gene list'].dropna().astype(str)).split(',')) - {''})).intersection(uni)
        obs = int(hit[drivers].sum())
        null = matched_null(hit.to_numpy(), decile.to_numpy(), decile[drivers].to_numpy(), n=N_NULL, seed=SEED)
        r3.append({'contrast': contrast, 'method': method, 'driver genes': len(drivers), 'flagged': obs,
                   'exposure': obs / len(drivers), 'null mean': float(null.mean() / len(drivers)),
                   'ratio': obs / max(null.mean(), 1e-9), 'p (≥)': float((1 + (null >= obs).sum()) / (N_NULL + 1))})
R3 = pd.DataFrame(r3)
R3.to_csv(tables / 'r3_exposure.csv', index=False)
display(R3.round(4))

## 7 · R4: the flag list, and verdicts

In [ ]:
flag_rows = []
for _, r in R1.iterrows():
    for g in [x for x in r['flagged genes'].split(', ') if x]:
        s = SPEC[r['species atlas']].loc[g]
        flag_rows.append({'list': r.list, 'gene': g, 'top non-PT type': s['top non-PT type'],
                          'PT CP10k': s['PT CP10k'], 'top non-PT CP10k': s['top non-PT CP10k'], 'ratio': s.ratio})
FLAGS = pd.DataFrame(flag_rows)
FLAGS.to_csv(tables / 'flags.csv', index=False)
display(FLAGS.groupby('list').size())
VERDICTS = pd.DataFrame([
    *[(f"R1 {r['list']}", f"{r['flagged']}/{r['assessable']} = {r['share']:.3f} (null {r['null mean']:.3f}): {'clean' if r['clean'] else 'not clean'}")
      for _, r in R1.iterrows()],
    *[(f"R2 edge enrichment tracks neighbour markers · {r['species']}", f"AUC {r['AUC of ε']:.3f}: {r['agree']}") for _, r in R2.iterrows()],
    *[(f"R3 {r['method']} · {r['contrast']}", f"{r['exposure']:.3f} vs {r['null mean']:.3f} (×{r['ratio']:.2f}, p {r['p (≥)']:.3g})") for _, r in R3.iterrows()],
    *[(f"post hoc suspect · {r['list']}", f"{r['suspect']}/{r['assessable']} = {r['suspect share']:.3f}") for _, r in SUSPECT.iterrows()],
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 8 · Figures (plots only)

In [ ]:
fig, ax = plt.subplots(figsize=(90 * MM, 80 * MM))
y = np.arange(len(R1))
ax.hlines(y, 0, R1['null 95%'], color=LIGHT, lw=3)
ax.scatter(R1.share, y, color=[SPECIES_COLOR[s] for s in R1['species atlas']], s=10, zorder=3)
ax.axvline(CLEAN_MAX, color=MUTED, lw=.5, ls=':')
ax.set_yticks(y, R1.list); ax.invert_yaxis()
ax.set_xlabel('share of neighbour-structure markers')
save(fig, '109a_lists')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(100 * MM, 40 * MM), sharey=True)
for ax, (sp, pair) in zip(axes, (('mouse', MOUSE), ('human', HUMAN))):
    e = eps[[f'hvm {s}' for s in pair]].mean(axis=1)
    if sp == 'human':
        e.index = [m2h.get(g, None) for g in e.index]
        e = e[e.index.notna()]
        e = e[~e.index.duplicated()]
    e = e.dropna()
    lab = SPEC[sp]['neighbour marker']
    common = e.index.intersection(lab.index)
    ax.boxplot([e[common][~lab[common]], e[common][lab[common]]], tick_labels=['other', 'neighbour marker'], showfliers=False)
    ax.set_title(sp)
axes[0].set_ylabel('edge enrichment ε')
save(fig, '109b_edge_vs_marker')
plt.show()

## 9 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Mouse lists use mouse symbols in the Kirita atlas, human lists human symbols in KPMP (feature_name); genes absent from an atlas are not assessable.'),
    ('reading', 'Expression deciles for the R1 null use notebook 62\'s log2 CP10k per species; for the AKI universe (mouse symbols without an inventory row) the atlas PT CP10k is used where the inventory is mostly missing.'),
    ('reading', 'R2 uses notebook 104\'s human vs mouse run, the mean ε over the species\' two specimens; human ε are keyed through notebook 90\'s mouse-to-human map.'),
    ('reading', 'R3 for human vs mouse counts a driver gene as flagged when it is a neighbour marker in either atlas; deciles use the mouse atlas PT CP10k.'),
    ('post hoc', 'Section 5.1 defines suspect genes (neighbour marker and edge-enriched in notebook 104) after R1 flagged 10–28% of most lists, close to the expression-matched null.'),
    ('reading', 'Conserved handoff genes are the genes of the (early, late) pairs present in both species\' pair lists of notebook 101.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '109_pt_neighbour_marker_filter', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'r1': R1.round(4).to_dict('records'), 'r2': R2.round(4).to_dict('records'), 'r3': R3.round(4).to_dict('records'),
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('guards', 'seconds')}, indent=2, default=str))